<a href="https://colab.research.google.com/github/syper4/AIColabs/blob/main/VideoGen/minimax/FreeVideo_MiniMax_H3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 FreeVideo: MiniMax H3 Video Generation on Google Colab

This notebook sets up and runs **[FreeVideo](https://github.com/FlashML-org/FreeVideo)** — a hardware-adaptive, low-VRAM inference engine for **MiniMax H3** powered by **Video DeltaNet (VDN)**.

### ✨ Key Capabilities:
- 🎥 **MiniMax H3 Model**: High-fidelity video generation from text prompts and image references.
- ⚡ **Low-Memory Adaptive Engine**: Weight streaming, chunked compute, and int8/FP8 quantization run MiniMax H3 within consumer GPU limits.
- 🖥️ **Integrated Creative Web UI**: Dedicated creative workspace inside ComfyUI (`?freevideo=launch`) designed for video creators.
- 🌐 **Accessible Public Link**: Uses a secure Cloudflare Tunnel to give you a public HTTPS URL accessible from anywhere.
- 🎞️ **Multimodal Support**: Text-to-Video (T2V), Image-to-Video (first & last frame keyframing), audio references, and community LoRA support.

> 💡 **Colab GPU Runtime**: Make sure you have chosen a GPU runtime before running (**Runtime → Change runtime type → T4 GPU / L4 / A100**).

## 1. 🔍 Check GPU & Runtime Hardware
Run this cell to verify your assigned GPU and verify available disk space.

In [ ]:
import subprocess, shutil

print("Checking GPU assignment...")
try:
    gpu_info = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
    print(gpu_info)
except Exception as e:
    raise RuntimeError(
        "❌ GPU not detected! Please go to Runtime -> Change runtime type -> select T4, L4, or A100 GPU."
    ) from e

total, used, free = shutil.disk_usage("/")
print(f"Storage: Total {total // (1024**3)} GB | Used {used // (1024**3)} GB | Free {free // (1024**3)} GB")


## 2. 📦 Install System Dependencies & Cloudflared Tunnel
Installs core system tools (`ffmpeg`, `aria2`, `curl`, `build-essential`) and the `cloudflared` tunnel client to provide a public Web UI link.

In [ ]:
import os

print("Updating and installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq --no-install-recommends ffmpeg aria2 git curl build-essential tar gzip ca-certificates

# Install Cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Downloading Cloudflared tunnel client...")
    !curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print("✅ Cloudflared installed.")
else:
    print("✅ Cloudflared already available.")


## 3. 📥 Clone FreeVideo Repository
Clones the official [FlashML-org/FreeVideo](https://github.com/FlashML-org/FreeVideo) repository to `/content/FreeVideo`.

In [ ]:
import os
from pathlib import Path

FREEVIDEO_ROOT = Path("/content/FreeVideo").resolve()

if FREEVIDEO_ROOT.exists():
    print(f"{FREEVIDEO_ROOT} already exists. Pulling latest code...")
    !git -C /content/FreeVideo pull
else:
    print(f"Cloning FreeVideo into {FREEVIDEO_ROOT}...")
    !git clone https://github.com/FlashML-org/FreeVideo.git /content/FreeVideo

%cd /content/FreeVideo
print("✅ FreeVideo repository ready.")


## 4. ⚙️ Automated Engine Setup & MiniMax H3 Model Download
Runs FreeVideo's unattended bootstrap setup (`./setup.sh`):
- Sets up `uv`, pinned Python environment, and GPU acceleration kernels (SageAttention).
- Automatically downloads the **OpenVDN MiniMax H3** model checkpoints, quantized weights (int8/FP8), text encoder, and latent upscaler.
- Deploys the ComfyUI backend and registers the FreeVideo creative workspace.

> ⏳ *Note*: The initial download and environment setup may take 8–15 minutes depending on connection speeds. Progress is displayed live below.

In [ ]:
%cd /content/FreeVideo
!chmod +x ./setup.sh ./freevideo

print("🚀 Starting FreeVideo setup (dependencies, Python env, & MiniMax H3 weights)...")
# --install-system-deps: auto installs missing deb packages
# --yes --accept-model-license: non-interactive agreement for Colab headless mode
# --plain: clean streaming log output for notebooks
!./setup.sh --install-system-deps --yes --accept-model-license --plain
print("✅ FreeVideo setup finished!")


## 5. 🌐 Launch FreeVideo Web UI with Public Cloudflare Link
Starts the FreeVideo server and creates a secure **Cloudflare Tunnel** so you can open the Web UI directly from your personal device/browser.

Click on the **Open FreeVideo Web UI** button or link printed below!

In [ ]:
import os, sys, subprocess, threading, time, socket, re
from pathlib import Path
from IPython.display import display, HTML

PORT = 8188
FREEVIDEO_ROOT = Path("/content/FreeVideo").resolve()
TOKEN_FILE = FREEVIDEO_ROOT / "launcher" / "access-token"

# 1. Helper to launch Cloudflare Tunnel
tunnel_url = None
def run_tunnel(port=8188):
    global tunnel_url
    print("⏳ Starting Cloudflare Tunnel...")
    cmd = ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in proc.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            print(f"\n🔗 Cloudflare Tunnel Established: {tunnel_url}")
            break

tunnel_thread = threading.Thread(target=run_tunnel, kwargs={"port": PORT}, daemon=True)
tunnel_thread.start()

# 2. Helper to display nice UI link once server is alive
def monitor_and_display_ui():
    # Wait for tunnel URL
    while tunnel_url is None:
        time.sleep(0.5)
    
    # Wait for port 8188 to open
    while True:
        try:
            with socket.create_connection(("127.0.0.1", PORT), timeout=1):
                break
        except (OSError, ConnectionRefusedError):
            time.sleep(1)
    
    token_val = ""
    if TOKEN_FILE.exists():
        token_val = TOKEN_FILE.read_text(encoding="utf-8").strip()
    
    web_link = f"{tunnel_url}/?freevideo=launch"
    if token_val:
        web_link = f"{tunnel_url}/?token={token_val}&freevideo=launch"
    
    display(HTML(f"""
    <div style="padding:20px; background:linear-gradient(135deg, #1f2937, #111827); border:1px solid #3b82f6; border-radius:12px; margin:20px 0; font-family:sans-serif;">
        <h2 style="color:#60a5fa; margin:0 0 10px 0;">✨ FreeVideo MiniMax H3 Web UI is Live!</h2>
        <p style="color:#e5e7eb; font-size:15px; margin:6px 0;">
            Click the button below to open the <b>FreeVideo Creative Workspace</b> in your browser:
        </p>
        <div style="margin:16px 0;">
            <a href="{web_link}" target="_blank" style="background-color:#2563eb; color:#ffffff; padding:12px 24px; text-decoration:none; border-radius:8px; font-weight:bold; font-size:16px; display:inline-block; box-shadow:0 4px 6px -1px rgba(0,0,0,0.3);">
                🚀 Open FreeVideo Web UI
            </a>
        </div>
        <p style="color:#9ca3af; font-size:13px; margin:4px 0;">
            Direct Link: <a href="{web_link}" target="_blank" style="color:#93c5fd;">{web_link}</a>
        </p>
    </div>
    """))

threading.Thread(target=monitor_and_display_ui, daemon=True).start()

# 3. Run the FreeVideo Server
%cd /content/FreeVideo
print("Starting FreeVideo server on 0.0.0.0:8188...")
!./freevideo server --listen 0.0.0.0 --port 8188


## 6. 🎬 (Optional) Standalone CLI Video Generation
If you prefer to generate videos directly via code or scripting without opening the Web UI, you can use the `./freevideo generate` command below.

In [ ]:
%cd /content/FreeVideo
from pathlib import Path
from IPython.display import Video, display

# Set your text prompt
prompt_text = "A cinematic aerial shot of misty pine mountains at sunrise, golden sunlight filtering through trees, realistic 4k"
Path("prompt.txt").write_text(prompt_text, encoding="utf-8")

output_path = Path("/content/generated_video.mp4").resolve()
print(f"Generating video for prompt: '{prompt_text}'...")

# Run generation (8-step base + 3-step refine)
!./freevideo generate --prompt-file prompt.txt --out {output_path} --width 1344 --height 768 --seconds 5

if output_path.exists():
    print("\n🎉 Video successfully generated! Displaying preview:")
    display(Video(str(output_path), embed=True))
else:
    print("Video file not found. Check command output above.")


## 💡 Tips & Performance Guidelines for Google Colab

- **VRAM Management**: On a standard T4 GPU (15–16 GB VRAM), FreeVideo automatically selects **int8** or **FP8** quantization and streaming. If you experience out-of-memory errors on high resolutions, try decreasing the resolution (`--width 960 --height 544`) or duration.
- **Two-Pass Preview**: In the Web UI, you can check the first-pass low-res preview first before running the full refinement pass.
- **Image-to-Video**: In the FreeVideo Web UI workspace, you can upload a first frame (and optional last frame) to generate controlled transitions and animations.
- **Stopping & Restarting**: To stop the server, click the square **Stop** button on Cell 5. To check status, run `!./freevideo status`.